# 06 Dashboard Data Preparation

This notebook prepares dashboard-ready datasets for Tableau based on the previous exploratory analysis and KPI engineering notebooks.

The goal is to recreate the final analytical tables, clean their structure, validate their shapes, and export them as CSV files for dashboard development.

This notebook does not perform new exploratory analysis. It focuses only on preparing final data outputs for visualization.

In [1]:
# Import core libraries
# Reference: 01_data_quality_check.ipynb - Step 1
import pandas as pd
import numpy as np
import os

# Set display options
# Reference: 01_data_quality_check.ipynb - Step 1
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Load all project datasets
# Reference: 01_data_quality_check.ipynb - Step 3
pharma_financials = pd.read_csv("pharma_companies_financials.csv")
drug_approvals = pd.read_csv("drug_approvals.csv")
clinical_trials = pd.read_csv("clinical_trials.csv")
biotech_funding = pd.read_csv("biotech_funding.csv")
disease_burden = pd.read_csv("disease_burden.csv")

# Convert date columns
# Reference: 01_data_quality_check.ipynb - Step 13
drug_approvals["approval_date"] = pd.to_datetime(drug_approvals["approval_date"])
clinical_trials["completion_date"] = pd.to_datetime(clinical_trials["completion_date"])
biotech_funding["date"] = pd.to_datetime(biotech_funding["date"])

# Create main analysis period datasets: 2010–2025
# Reference: 02_financial_performance_eda.ipynb
pharma_financials_main = pharma_financials[
    (pharma_financials["year"] >= 2010) & (pharma_financials["year"] <= 2025)
]

drug_approvals_main = drug_approvals[
    (drug_approvals["year"] >= 2010) & (drug_approvals["year"] <= 2025)
]

clinical_trials_main = clinical_trials[
    (clinical_trials["year"] >= 2010) & (clinical_trials["year"] <= 2025)
]

biotech_funding_main = biotech_funding[
    (biotech_funding["year"] >= 2010) & (biotech_funding["year"] <= 2025)
]

disease_burden_main = disease_burden[
    (disease_burden["year"] >= 2010) & (disease_burden["year"] <= 2025)
]

## Step 2: Validate Loaded Datasets

This step checks the shape and year range of all datasets used for dashboard data preparation.

In [2]:
dataset_check = pd.DataFrame({
    "dataset": [
        "pharma_financials_main",
        "drug_approvals_main",
        "clinical_trials_main",
        "biotech_funding_main",
        "disease_burden_main"
    ],
    "rows": [
        pharma_financials_main.shape[0],
        drug_approvals_main.shape[0],
        clinical_trials_main.shape[0],
        biotech_funding_main.shape[0],
        disease_burden_main.shape[0]
    ],
    "columns": [
        pharma_financials_main.shape[1],
        drug_approvals_main.shape[1],
        clinical_trials_main.shape[1],
        biotech_funding_main.shape[1],
        disease_burden_main.shape[1]
    ],
    "min_year": [
        pharma_financials_main["year"].min(),
        drug_approvals_main["year"].min(),
        clinical_trials_main["year"].min(),
        biotech_funding_main["year"].min(),
        disease_burden_main["year"].min()
    ],
    "max_year": [
        pharma_financials_main["year"].max(),
        drug_approvals_main["year"].max(),
        clinical_trials_main["year"].max(),
        biotech_funding_main["year"].max(),
        disease_burden_main["year"].max()
    ]
})

dataset_check

,dataset,rows,columns,min_year,max_year
0,pharma_financials_main,459,10,2010,2025
1,drug_approvals_main,715,12,2010,2025
2,clinical_trials_main,595,12,2010,2025
3,biotech_funding_main,1201,10,2010,2025
4,disease_burden_main,3110,5,2010,2025


## Step 3: Create Company KPI Dashboard Table

This step creates a company-level dashboard table by combining financial performance metrics with drug approval output metrics.

This table will support Tableau views related to company performance, R&D investment, approval productivity, commercial output, and segment-level comparison.

In [4]:
# Create company-level financial KPIs
company_financial_kpis = (
    pharma_financials_main
    .groupby(["company_name", "ticker", "segment"])
    .agg(
        total_revenue_usd_bn=("revenue_usd_bn", "sum"),
        total_rd_spend_usd_bn=("rd_spend_usd_bn", "sum"),
        avg_operating_margin_pct=("operating_margin_pct", "mean"),
        avg_pipeline_size=("pipeline_size_est", "mean")
    )
    .reset_index()
)

company_financial_kpis["rd_intensity_pct"] = (
    company_financial_kpis["total_rd_spend_usd_bn"] /
    company_financial_kpis["total_revenue_usd_bn"]
) * 100


# Standardize company names in approvals dataset before merging
drug_approvals_company_clean = drug_approvals_main.copy()

company_name_mapping = {
    "J&J": "Johnson & Johnson",
    "BMS": "Bristol-Myers Squibb"
}

drug_approvals_company_clean["sponsor_company_clean"] = (
    drug_approvals_company_clean["sponsor_company"]
    .replace(company_name_mapping)
)


# Create company-level approval KPIs
company_approval_kpis = (
    drug_approvals_company_clean
    .groupby("sponsor_company_clean")
    .agg(
        total_approvals=("approval_id", "count"),
        blockbuster_count=("is_blockbuster", "sum"),
        mega_blockbuster_count=("is_mega_blockbuster", "sum"),
        estimated_commercial_output_usd_bn=("peak_sales_usd_bn_est", "sum"),
        avg_peak_sales_usd_bn=("peak_sales_usd_bn_est", "mean")
    )
    .reset_index()
)


# Merge financial and approval KPIs
company_kpis_dashboard = company_financial_kpis.merge(
    company_approval_kpis,
    left_on="company_name",
    right_on="sponsor_company_clean",
    how="left"
)

approval_columns = [
    "total_approvals",
    "blockbuster_count",
    "mega_blockbuster_count",
    "estimated_commercial_output_usd_bn",
    "avg_peak_sales_usd_bn"
]

company_kpis_dashboard[approval_columns] = (
    company_kpis_dashboard[approval_columns]
    .fillna(0)
)

company_kpis_dashboard = company_kpis_dashboard.drop(
    columns=["sponsor_company_clean"]
)


# Create productivity KPIs
company_kpis_dashboard["approval_productivity_per_bn_rd"] = (
    company_kpis_dashboard["total_approvals"] /
    company_kpis_dashboard["total_rd_spend_usd_bn"]
)

company_kpis_dashboard["commercial_output_per_bn_rd"] = (
    company_kpis_dashboard["estimated_commercial_output_usd_bn"] /
    company_kpis_dashboard["total_rd_spend_usd_bn"]
)

company_kpis_dashboard["blockbuster_rate_pct"] = np.where(
    company_kpis_dashboard["total_approvals"] > 0,
    (company_kpis_dashboard["blockbuster_count"] /
     company_kpis_dashboard["total_approvals"]) * 100,
    0
)


# Round numeric columns for dashboard readability
numeric_columns = company_kpis_dashboard.select_dtypes(include=["float64", "int64"]).columns
company_kpis_dashboard[numeric_columns] = company_kpis_dashboard[numeric_columns].round(2)


# Sort by total revenue
company_kpis_dashboard = (
    company_kpis_dashboard
    .sort_values("total_revenue_usd_bn", ascending=False)
    .reset_index(drop=True)
)

company_kpis_dashboard.head(10)

,company_name,ticker,segment,total_revenue_usd_bn,total_rd_spend_usd_bn,avg_operating_margin_pct,avg_pipeline_size,rd_intensity_pct,total_approvals,blockbuster_count,mega_blockbuster_count,estimated_commercial_output_usd_bn,avg_peak_sales_usd_bn,approval_productivity_per_bn_rd,commercial_output_per_bn_rd,blockbuster_rate_pct
0,Johnson & Johnson,JNJ,diversified_pharma,1257.64,150.92,21.38,79.44,12.0,28.0,4.0,0.0,18.62,0.66,0.19,0.12,14.29
1,Pfizer,PFE,big_pharma,959.64,172.71,27.46,109.81,18.0,20.0,4.0,1.0,27.67,1.38,0.12,0.16,20.00
2,Roche,ROG,big_pharma,915.30,164.75,27.28,108.56,18.0,23.0,7.0,3.0,39.92,1.74,0.14,0.24,30.43
3,Novartis,NVS,big_pharma,804.72,144.84,27.01,104.56,18.0,27.0,5.0,1.0,18.30,0.68,0.19,0.13,18.52
4,Merck,MRK,big_pharma,781.19,140.60,28.11,103.19,18.0,26.0,9.0,2.0,55.47,2.13,0.18,0.39,34.62
5,Bayer,BAYN,diversified_pharma,779.43,93.55,15.68,68.88,12.0,19.0,1.0,0.0,7.41,0.39,0.20,0.08,5.26
6,Sanofi,SAN,big_pharma,596.13,107.29,28.19,95.56,18.0,20.0,5.0,1.0,15.82,0.79,0.19,0.15,25.00
7,AstraZeneca,AZN,big_pharma,534.71,96.25,27.62,91.56,18.0,26.0,5.0,0.0,17.96,0.69,0.27,0.19,19.23
8,Abbott Laboratories,ABT,diversified_pharma,519.25,62.32,21.68,60.62,12.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00
9,AbbVie,ABBV,big_pharma,514.18,92.55,28.82,95.85,18.0,22.0,4.0,2.0,22.16,1.01,0.24,0.24,18.18


## Step 4: Create Therapy Area Innovation Gap Dashboard Table

This step creates a therapy-area-level dashboard table by combining drug approval activity, clinical trial activity, and disease burden.

This table will support Tableau views related to innovation concentration, disease burden alignment, under-prioritized therapy areas, and Innovation Gap Score.

In [5]:
# Create approval KPIs by therapy area
approval_kpis_by_therapy = (
    drug_approvals_main
    .groupby("therapy_area")
    .agg(
        approval_count=("approval_id", "count"),
        estimated_commercial_output_usd_bn=("peak_sales_usd_bn_est", "sum"),
        avg_peak_sales_usd_bn=("peak_sales_usd_bn_est", "mean"),
        blockbuster_count=("is_blockbuster", "sum"),
        mega_blockbuster_count=("is_mega_blockbuster", "sum")
    )
    .reset_index()
)


# Create clinical trial KPIs by therapy area
trial_kpis_by_therapy = (
    clinical_trials_main
    .groupby("therapy_area")
    .agg(
        trial_count=("trial_id", "count"),
        successful_trials=("is_success", "sum"),
        failed_trials=("is_failure", "sum"),
        trial_success_rate_pct=("is_success", "mean"),
        avg_stock_impact_pct=("estimated_stock_impact_pct", "mean")
    )
    .reset_index()
)

trial_kpis_by_therapy["trial_success_rate_pct"] = (
    trial_kpis_by_therapy["trial_success_rate_pct"] * 100
)


# Combine approval and trial KPIs
therapy_innovation_kpis = approval_kpis_by_therapy.merge(
    trial_kpis_by_therapy,
    on="therapy_area",
    how="outer"
).fillna(0)


# Map disease categories to therapy areas
disease_to_therapy_mapping = {
    "Cancer": "oncology",
    "Cardiovascular disease": "cardiovascular",
    "Stroke": "cardiovascular",
    "Neurological disorders": "neurology",
    "Alzheimer's & dementias": "neurology",
    "Mental disorders": "psychiatry",
    "Diabetes": "metabolic",
    "Chronic respiratory": "respiratory",
    "Lower respiratory infections": "infectious",
    "Tuberculosis": "infectious",
    "Malaria": "infectious",
    "HIV/AIDS": "infectious",
    "COVID-19": "infectious",
    "Diarrheal diseases": "gastrointestinal",
    "Liver disease": "gastrointestinal",
    "Kidney disease": "gastrointestinal",
    "Neonatal disorders": "rare_disease",
    "Maternal disorders": "rare_disease",
    "Self-harm": "psychiatry"
}

disease_burden_mapped = disease_burden_main.copy()

disease_burden_mapped["therapy_area"] = disease_burden_mapped["disease"].map(
    disease_to_therapy_mapping
)


# Create disease burden KPIs by therapy area
therapy_burden_kpis = (
    disease_burden_mapped
    .dropna(subset=["therapy_area"])
    .groupby("therapy_area")
    .agg(
        mapped_disease_count=("disease", "nunique"),
        avg_dalys_millions=("dalys_millions", "mean"),
        total_dalys_millions=("dalys_millions", "sum"),
        avg_global_dalys_millions=("global_dalys_millions", "mean")
    )
    .reset_index()
)


# Combine innovation and burden KPIs
therapy_area_innovation_gap_dashboard = therapy_innovation_kpis.merge(
    therapy_burden_kpis,
    on="therapy_area",
    how="inner"
)


# Create ranks
therapy_area_innovation_gap_dashboard["disease_burden_rank"] = (
    therapy_area_innovation_gap_dashboard["avg_global_dalys_millions"]
    .rank(ascending=False, method="dense")
)

therapy_area_innovation_gap_dashboard["approval_rank"] = (
    therapy_area_innovation_gap_dashboard["approval_count"]
    .rank(ascending=False, method="dense")
)

therapy_area_innovation_gap_dashboard["commercial_output_rank"] = (
    therapy_area_innovation_gap_dashboard["estimated_commercial_output_usd_bn"]
    .rank(ascending=False, method="dense")
)

therapy_area_innovation_gap_dashboard["trial_activity_rank"] = (
    therapy_area_innovation_gap_dashboard["trial_count"]
    .rank(ascending=False, method="dense")
)


# Create Innovation Rank and Innovation Gap Score
therapy_area_innovation_gap_dashboard["innovation_rank"] = (
    therapy_area_innovation_gap_dashboard[
        ["approval_rank", "commercial_output_rank", "trial_activity_rank"]
    ].mean(axis=1)
)

therapy_area_innovation_gap_dashboard["innovation_gap_score"] = (
    therapy_area_innovation_gap_dashboard["innovation_rank"] -
    therapy_area_innovation_gap_dashboard["disease_burden_rank"]
)


# Round numeric columns
numeric_columns = therapy_area_innovation_gap_dashboard.select_dtypes(
    include=["float64", "int64"]
).columns

therapy_area_innovation_gap_dashboard[numeric_columns] = (
    therapy_area_innovation_gap_dashboard[numeric_columns].round(2)
)


# Sort by Innovation Gap Score
therapy_area_innovation_gap_dashboard = (
    therapy_area_innovation_gap_dashboard
    .sort_values("innovation_gap_score", ascending=False)
    .reset_index(drop=True)
)

therapy_area_innovation_gap_dashboard

,therapy_area,approval_count,estimated_commercial_output_usd_bn,avg_peak_sales_usd_bn,blockbuster_count,mega_blockbuster_count,trial_count,successful_trials,failed_trials,trial_success_rate_pct,avg_stock_impact_pct,mapped_disease_count,avg_dalys_millions,total_dalys_millions,avg_global_dalys_millions,disease_burden_rank,approval_rank,commercial_output_rank,trial_activity_rank,innovation_rank,innovation_gap_score
0,cardiovascular,49,58.48,1.19,11,3,67.0,54.0,9.0,80.60,3.18,2,26.35,8433.05,263.80,1.0,5.0,6.0,2.0,4.33,3.33
1,respiratory,20,15.62,0.78,3,1,23.0,18.0,4.0,78.26,5.42,1,10.24,1639.00,102.19,4.0,7.0,7.0,7.0,7.00,3.00
2,psychiatry,20,9.68,0.48,2,0,17.0,7.0,10.0,41.18,-3.72,2,10.21,3268.07,101.28,5.0,7.0,8.0,8.0,7.67,2.67
3,neurology,94,90.50,0.96,22,4,62.0,17.0,38.0,27.42,-7.30,2,14.13,4523.09,141.09,3.0,2.0,4.0,3.0,3.00,0.00
4,gastrointestinal,9,3.93,0.44,1,0,23.0,15.0,5.0,65.22,2.70,3,5.05,2424.76,50.44,9.0,8.0,9.0,7.0,8.00,-1.00
5,oncology,272,227.09,0.83,49,10,179.0,62.0,82.0,34.64,-4.85,1,24.65,3944.53,246.91,2.0,1.0,1.0,1.0,1.00,-1.00
6,metabolic,38,90.15,2.37,11,5,47.0,29.0,5.0,61.70,3.21,1,7.56,1210.00,75.38,7.0,6.0,5.0,5.0,5.33,-1.67
7,rare_disease,68,93.56,1.38,19,5,45.0,24.0,17.0,53.33,-0.51,2,9.49,3036.43,92.06,6.0,3.0,3.0,6.0,4.00,-2.00
8,infectious,57,149.48,2.62,16,7,58.0,43.0,5.0,74.14,5.03,5,6.69,4747.93,60.81,8.0,4.0,2.0,4.0,3.33,-4.67


## Step 5: Create Yearly Trends Dashboard Table

This step creates a year-level dashboard table by combining financial, approval, clinical trial, biotech funding, and disease burden metrics.

This table will support Tableau views related to industry trends, revenue growth, R&D spending, approval activity, trial activity, funding cycles, and disease burden over time.

In [6]:
# Financial trends by year
yearly_financials = (
    pharma_financials_main
    .groupby("year")
    .agg(
        total_revenue_usd_bn=("revenue_usd_bn", "sum"),
        total_rd_spend_usd_bn=("rd_spend_usd_bn", "sum"),
        avg_operating_margin_pct=("operating_margin_pct", "mean")
    )
    .reset_index()
)

yearly_financials["rd_intensity_pct"] = (
    yearly_financials["total_rd_spend_usd_bn"] /
    yearly_financials["total_revenue_usd_bn"]
) * 100


# Drug approval trends by year
yearly_approvals = (
    drug_approvals_main
    .groupby("year")
    .agg(
        approval_count=("approval_id", "count"),
        blockbuster_count=("is_blockbuster", "sum"),
        mega_blockbuster_count=("is_mega_blockbuster", "sum"),
        estimated_commercial_output_usd_bn=("peak_sales_usd_bn_est", "sum")
    )
    .reset_index()
)


# Clinical trial trends by year
yearly_trials = (
    clinical_trials_main
    .groupby("year")
    .agg(
        trial_count=("trial_id", "count"),
        successful_trials=("is_success", "sum"),
        failed_trials=("is_failure", "sum"),
        avg_stock_impact_pct=("estimated_stock_impact_pct", "mean")
    )
    .reset_index()
)

yearly_trials["trial_success_rate_pct"] = (
    yearly_trials["successful_trials"] /
    yearly_trials["trial_count"]
) * 100


# Biotech funding trends by year
yearly_funding = (
    biotech_funding_main
    .groupby("year")
    .agg(
        funding_deal_count=("deal_id", "count"),
        total_funding_usd_bn=("value_usd_bn", "sum"),
        avg_deal_value_usd_bn=("value_usd_bn", "mean"),
        megadeal_count=("is_megadeal", "sum")
    )
    .reset_index()
)


# Disease burden trends by year
yearly_burden = (
    disease_burden_main
    .groupby("year")
    .agg(
        total_dalys_millions=("dalys_millions", "sum"),
        avg_dalys_millions=("dalys_millions", "mean")
    )
    .reset_index()
)


# Combine all yearly trend tables
yearly_trends_dashboard = (
    yearly_financials
    .merge(yearly_approvals, on="year", how="outer")
    .merge(yearly_trials, on="year", how="outer")
    .merge(yearly_funding, on="year", how="outer")
    .merge(yearly_burden, on="year", how="outer")
    .sort_values("year")
    .reset_index(drop=True)
)


# Round numeric columns
numeric_columns = yearly_trends_dashboard.select_dtypes(
    include=["float64", "int64"]
).columns

yearly_trends_dashboard[numeric_columns] = (
    yearly_trends_dashboard[numeric_columns].round(2)
)

yearly_trends_dashboard

,year,total_revenue_usd_bn,total_rd_spend_usd_bn,avg_operating_margin_pct,rd_intensity_pct,approval_count,blockbuster_count,mega_blockbuster_count,estimated_commercial_output_usd_bn,trial_count,successful_trials,failed_trials,avg_stock_impact_pct,trial_success_rate_pct,funding_deal_count,total_funding_usd_bn,avg_deal_value_usd_bn,megadeal_count,total_dalys_millions,avg_dalys_millions
0,2010,640.24,102.78,25.58,16.05,24,5,0,19.62,34,15,15,-1.91,44.12,35,16.17,0.46,0,2112.62,11.12
1,2011,643.01,103.49,25.83,16.09,32,7,2,27.81,42,23,8,-0.11,54.76,45,51.77,1.15,2,2091.57,11.01
2,2012,643.09,103.76,26.07,16.13,41,8,3,48.36,41,22,16,-0.69,53.66,50,64.84,1.30,1,2105.60,11.08
3,2013,662.43,107.49,26.60,16.23,29,7,3,47.67,42,20,12,0.04,47.62,55,58.60,1.07,2,2113.87,11.13
4,2014,674.78,109.63,25.39,16.25,44,6,3,77.37,32,19,8,2.26,59.38,62,174.81,2.82,4,2103.59,11.07
5,2015,688.35,111.98,25.82,16.27,48,10,3,44.66,36,21,12,-1.41,58.33,71,155.18,2.19,5,2089.48,11.00
6,2016,716.36,116.69,26.23,16.29,23,7,1,27.06,31,13,14,-4.12,41.94,66,54.72,0.83,1,2097.40,11.04
7,2017,732.36,118.88,26.60,16.23,52,20,3,83.63,41,24,10,-0.17,58.54,75,71.00,0.95,2,2092.33,11.01
8,2018,753.19,122.46,26.36,16.26,59,12,2,56.26,36,20,14,-2.30,55.56,88,48.24,0.55,0,2101.71,11.06
9,2019,785.16,128.23,26.42,16.33,50,7,2,39.88,33,19,10,-0.67,57.58,72,282.49,3.92,6,2093.35,10.47


## Step 6: Create Funding Deal Type Dashboard Table

This step creates a deal-type-level biotech funding table for Tableau.

This table will support dashboard views comparing VC, M&A, IPO, licensing, and other deal categories by deal count, total funding value, average deal value, and megadeal count.

In [7]:
funding_deal_type_dashboard = (
    biotech_funding_main
    .groupby("deal_type")
    .agg(
        deal_count=("deal_id", "count"),
        total_funding_usd_bn=("value_usd_bn", "sum"),
        avg_deal_value_usd_bn=("value_usd_bn", "mean"),
        megadeal_count=("is_megadeal", "sum")
    )
    .sort_values("total_funding_usd_bn", ascending=False)
    .reset_index()
)

numeric_columns = funding_deal_type_dashboard.select_dtypes(
    include=["float64", "int64"]
).columns

funding_deal_type_dashboard[numeric_columns] = (
    funding_deal_type_dashboard[numeric_columns].round(2)
)

funding_deal_type_dashboard

,deal_type,deal_count,total_funding_usd_bn,avg_deal_value_usd_bn,megadeal_count
0,M&A,192,1604.64,8.36,47
1,VC,465,105.58,0.23,0
2,IPO,134,71.05,0.53,0
3,Series_B,231,56.82,0.25,0
4,Series_C,118,29.52,0.25,0
5,Crossover,61,13.45,0.22,0


## Step 7: Create Megadeal Summary Dashboard Table

This step creates a summary table comparing all deals, megadeals, and non-megadeals.

This table will support Tableau views related to funding concentration and the disproportionate impact of megadeals on total biotech funding value.

In [8]:
megadeal_summary_dashboard = pd.DataFrame({
    "category": ["All Deals", "Megadeals", "Non-Megadeals"],
    "deal_count": [
        biotech_funding_main["deal_id"].count(),
        biotech_funding_main[biotech_funding_main["is_megadeal"] == 1]["deal_id"].count(),
        biotech_funding_main[biotech_funding_main["is_megadeal"] == 0]["deal_id"].count()
    ],
    "total_funding_usd_bn": [
        biotech_funding_main["value_usd_bn"].sum(),
        biotech_funding_main[biotech_funding_main["is_megadeal"] == 1]["value_usd_bn"].sum(),
        biotech_funding_main[biotech_funding_main["is_megadeal"] == 0]["value_usd_bn"].sum()
    ]
})

megadeal_summary_dashboard["deal_share_pct"] = (
    megadeal_summary_dashboard["deal_count"] /
    megadeal_summary_dashboard.loc[0, "deal_count"]
) * 100

megadeal_summary_dashboard["funding_share_pct"] = (
    megadeal_summary_dashboard["total_funding_usd_bn"] /
    megadeal_summary_dashboard.loc[0, "total_funding_usd_bn"]
) * 100

numeric_columns = megadeal_summary_dashboard.select_dtypes(
    include=["float64", "int64"]
).columns

megadeal_summary_dashboard[numeric_columns] = (
    megadeal_summary_dashboard[numeric_columns].round(2)
)

megadeal_summary_dashboard

,category,deal_count,total_funding_usd_bn,deal_share_pct,funding_share_pct
0,All Deals,1201,1881.06,100.00,100.00
1,Megadeals,47,1316.43,3.91,69.98
2,Non-Megadeals,1154,564.64,96.09,30.02


## Step 8: Create Regional Disease Burden Dashboard Table

This step creates a region-level disease burden table for Tableau.

This table will support dashboard views related to regional disease burden concentration and global health need.

In [9]:
regional_disease_burden_dashboard = (
    disease_burden_main
    .groupby("region")
    .agg(
        record_count=("region", "count"),
        avg_dalys_millions=("dalys_millions", "mean"),
        total_dalys_millions=("dalys_millions", "sum")
    )
    .sort_values("total_dalys_millions", ascending=False)
    .reset_index()
)

numeric_columns = regional_disease_burden_dashboard.select_dtypes(
    include=["float64", "int64"]
).columns

regional_disease_burden_dashboard[numeric_columns] = (
    regional_disease_burden_dashboard[numeric_columns].round(2)
)

regional_disease_burden_dashboard

,region,record_count,avg_dalys_millions,total_dalys_millions
0,Africa,311,24.29,7555.45
1,India,311,21.72,6754.21
2,China,311,19.68,6118.99
3,Southeast Asia,311,10.87,3379.20
4,Middle East,311,7.64,2375.03
5,Latin America,311,7.62,2371.36
6,Western Europe,311,7.41,2305.36
7,North America,311,5.97,1856.59
8,Eastern Europe,311,4.37,1358.28
9,Oceania,311,1.08,336.43


## Step 9: Validate Dashboard-Ready Tables

This step checks the shape of each dashboard-ready table before exporting them as CSV files.

Only final dashboard tables will be exported. Intermediate analysis tables will remain inside the notebooks.

In [10]:
dashboard_table_check = pd.DataFrame({
    "table_name": [
        "company_kpis_dashboard",
        "therapy_area_innovation_gap_dashboard",
        "yearly_trends_dashboard",
        "funding_deal_type_dashboard",
        "megadeal_summary_dashboard",
        "regional_disease_burden_dashboard"
    ],
    "rows": [
        company_kpis_dashboard.shape[0],
        therapy_area_innovation_gap_dashboard.shape[0],
        yearly_trends_dashboard.shape[0],
        funding_deal_type_dashboard.shape[0],
        megadeal_summary_dashboard.shape[0],
        regional_disease_burden_dashboard.shape[0]
    ],
    "columns": [
        company_kpis_dashboard.shape[1],
        therapy_area_innovation_gap_dashboard.shape[1],
        yearly_trends_dashboard.shape[1],
        funding_deal_type_dashboard.shape[1],
        megadeal_summary_dashboard.shape[1],
        regional_disease_burden_dashboard.shape[1]
    ]
})

dashboard_table_check

,table_name,rows,columns
0,company_kpis_dashboard,30,16
1,therapy_area_innovation_gap_dashboard,9,21
2,yearly_trends_dashboard,16,20
3,funding_deal_type_dashboard,6,5
4,megadeal_summary_dashboard,3,5
5,regional_disease_burden_dashboard,10,4


## Step 10: Export Dashboard-Ready Tables

This step exports the final dashboard-ready tables as CSV files into an `exports` folder.

These CSV files will be used as Tableau data sources.

In [12]:
# Create exports folder
os.makedirs("exports", exist_ok=True)

# Export dashboard-ready tables
company_kpis_dashboard.to_csv(
    "exports/company_kpis_dashboard.csv",
    index=False
)

therapy_area_innovation_gap_dashboard.to_csv(
    "exports/therapy_area_innovation_gap_dashboard.csv",
    index=False
)

yearly_trends_dashboard.to_csv(
    "exports/yearly_trends_dashboard.csv",
    index=False
)

funding_deal_type_dashboard.to_csv(
    "exports/funding_deal_type_dashboard.csv",
    index=False
)

megadeal_summary_dashboard.to_csv(
    "exports/megadeal_summary_dashboard.csv",
    index=False
)

regional_disease_burden_dashboard.to_csv(
    "exports/regional_disease_burden_dashboard.csv",
    index=False
)

## Step 11: Validate Exported CSV Files

This step checks whether the dashboard-ready CSV files were successfully exported into the `exports` folder.

In [13]:
exported_files = os.listdir("exports")

exported_files

['.ipynb_checkpoints',
 'company_kpis_dashboard.csv',
 'funding_deal_type_dashboard.csv',
 'megadeal_summary_dashboard.csv',
 'regional_disease_burden_dashboard.csv',
 'therapy_area_innovation_gap_dashboard.csv',
 'yearly_trends_dashboard.csv']

## Step 12: Re-Read Exported CSV Files

This step re-reads the exported CSV files to confirm that they can be loaded successfully and that their shapes match the dashboard-ready tables.

In [14]:
export_check = pd.DataFrame({
    "file_name": [
        "company_kpis_dashboard.csv",
        "therapy_area_innovation_gap_dashboard.csv",
        "yearly_trends_dashboard.csv",
        "funding_deal_type_dashboard.csv",
        "megadeal_summary_dashboard.csv",
        "regional_disease_burden_dashboard.csv"
    ],
    "rows_after_export": [
        pd.read_csv("exports/company_kpis_dashboard.csv").shape[0],
        pd.read_csv("exports/therapy_area_innovation_gap_dashboard.csv").shape[0],
        pd.read_csv("exports/yearly_trends_dashboard.csv").shape[0],
        pd.read_csv("exports/funding_deal_type_dashboard.csv").shape[0],
        pd.read_csv("exports/megadeal_summary_dashboard.csv").shape[0],
        pd.read_csv("exports/regional_disease_burden_dashboard.csv").shape[0]
    ],
    "columns_after_export": [
        pd.read_csv("exports/company_kpis_dashboard.csv").shape[1],
        pd.read_csv("exports/therapy_area_innovation_gap_dashboard.csv").shape[1],
        pd.read_csv("exports/yearly_trends_dashboard.csv").shape[1],
        pd.read_csv("exports/funding_deal_type_dashboard.csv").shape[1],
        pd.read_csv("exports/megadeal_summary_dashboard.csv").shape[1],
        pd.read_csv("exports/regional_disease_burden_dashboard.csv").shape[1]
    ]
})

export_check

,file_name,rows_after_export,columns_after_export
0,company_kpis_dashboard.csv,30,16
1,therapy_area_innovation_gap_dashboard.csv,9,21
2,yearly_trends_dashboard.csv,16,20
3,funding_deal_type_dashboard.csv,6,5
4,megadeal_summary_dashboard.csv,3,5
5,regional_disease_burden_dashboard.csv,10,4


# Notebook Summary

This notebook prepared the final dashboard-ready datasets for Tableau.

Exported CSV files:

- company_kpis_dashboard.csv
- therapy_area_innovation_gap_dashboard.csv
- yearly_trends_dashboard.csv
- funding_deal_type_dashboard.csv
- megadeal_summary_dashboard.csv
- regional_disease_burden_dashboard.csv

These files are stored in the `exports` folder and will be used as Tableau data sources.

The dashboard preparation stage is complete.